# 8.2 端侧训练优化

在端侧设备上进行模型微调，需要解决显存不足、计算能力有限和训练稳定性等挑战。端侧训练优化是让大模型在资源受限设备上可训练的关键技术。

## 什么是端侧训练优化？

端侧训练优化是在显存和算力受限的端侧设备上高效训练模型的技术集合，包括：
- **混合精度训练**：使用FP16/BF16减少显存占用，利用Tensor Core加速
- **梯度检查点**：用计算换显存，只保存部分中间激活
- **8bit优化器**：将优化器状态量化为8bit，减少75%优化器显存
- **CPU Offloading**：将优化器状态卸载到CPU内存
- **DeepSpeed ZeRO**：分阶段划分优化器状态、梯度和参数
- **量化感知训练(QAT)**：训练中插入伪量化，让模型适应量化精度损失
- **知识蒸馏**：用大模型指导小模型训练

## 为什么需要端侧训练优化？

1. **显存瓶颈**：7B模型全量微调需要约112GB显存（参数+梯度+优化器状态），远超端侧设备
2. **算力限制**：端侧GPU/CPU算力远低于数据中心GPU
3. **训练稳定性**：低精度训练容易出现梯度溢出和NaN

## 端侧训练优化的数学原理

**显存分解**：训练显存 = 参数 + 梯度 + 优化器状态 + 激活值
$$M_{\text{total}} = 2P + 2P + (8+8)P + A = 20P + A$$
其中$P$为参数量，$A$为激活值大小（AdamW: m和v各4字节/参数）

7B模型：$20 \times 7B = 140$ GB（不含激活值）

**混合精度**：前向用FP16，梯度用FP16，优化器状态用FP32（master weights）
$$M_{\text{AMP}} = 2P + 2P + 12P + A_{\text{FP16}} = 16P + A_{\text{FP16}}$$

**梯度检查点**：只保存每$k$层的激活，其余重新计算
$$A_{\text{ckpt}} = \frac{A}{k} + \frac{k-1}{k} \cdot A_{\text{recompute}}$$
显存降低约$k$倍，计算量增加约1次前向

**8bit优化器**：将Adam的$m$和$v$量化为8bit
$$M_{\text{8bit}} = 2P + 2P + 2P + A = 6P + A$$
优化器状态从16字节/参数降至2字节/参数

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import gc
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### 混合精度训练

#### 什么是混合精度训练？

使用低精度（FP16/BF16）进行前向和反向传播，同时维护FP32的master weights用于参数更新。兼顾训练速度和数值稳定性。

#### 为什么用混合精度训练？

1. **显存减半**：FP16权重和激活值各占FP32的一半
2. **计算加速**：NVIDIA GPU的Tensor Core对FP16矩阵乘法加速2-8倍
3. **带宽减半**：梯度通信量减半（分布式训练中尤为重要）

#### 混合精度训练的数学原理

前向传播用FP16：
$$h_{\text{FP16}} = \text{cast}_{\text{FP16}}(W_{\text{FP32}}) \cdot x_{\text{FP16}}$$

损失缩放（Loss Scaling）防止梯度下溢：
$$L_{\text{scaled}} = L \cdot s, \quad g_{\text{scaled}} = s \cdot g$$
$$g_{\text{true}} = g_{\text{scaled}} / s$$

参数更新用FP32（master weights）：
$$W_{\text{FP32}}^{(t+1)} = W_{\text{FP32}}^{(t)} - \eta \cdot g_{\text{FP32}}$$

#### FP16 vs BF16

| 特性 | FP16 | BF16 |
|------|------|------|
| 指数位 | 5位 | 8位 |
| 尾数位 | 10位 | 7位 |
| 动态范围 | $6 \times 10^{-8}$ ~ 65504 | $1.2 \times 10^{-38}$ ~ $3.4 \times 10^{38}$ |
| 需要Loss Scaling | 是 | 否 |
| 硬件支持 | NVIDIA GPU (Volta+) | NVIDIA GPU (Ampere+), CPU, TPU |
| 推荐场景 | GPU训练 | CPU训练/Ampere+GPU |

关键区别：BF16的动态范围与FP32相同，因此不需要loss scaling，训练更稳定。
CPU上autocast只支持BF16，不支持FP16。

In [ ]:
class MixedPrecisionTrainer:
    """混合精度训练器 - 产业级实现
    CPU上应使用bfloat16，GPU上推荐float16(Tensor Core加速)或bfloat16(Ampere+更稳定)"""
    def __init__(self, model, lr=1e-3, weight_decay=0.01, use_bf16=True, grad_clip=1.0):
        self.model = model
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
        self.grad_clip = grad_clip
        self.use_bf16 = use_bf16

        if torch.cuda.is_available():
            self.device_type = 'cuda'
            self.dtype = torch.bfloat16 if use_bf16 else torch.float16
            self.scaler = torch.amp.GradScaler('cuda', enabled=(not use_bf16))
        else:
            self.device_type = 'cpu'
            self.dtype = torch.bfloat16
            self.scaler = torch.amp.GradScaler('cuda', enabled=False)

    def train_step(self, x, y):
        self.model.train()
        self.optimizer.zero_grad()
        with torch.autocast(device_type=self.device_type, dtype=self.dtype):
            loss = F.cross_entropy(self.model(x), y)

        if self.device_type == 'cuda' and not self.use_bf16:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
        return loss.item()

    def evaluate(self, x, y):
        self.model.eval()
        with torch.no_grad(), torch.autocast(device_type=self.device_type, dtype=self.dtype):
            logits = self.model(x)
            loss = F.cross_entropy(logits, y)
            acc = (logits.argmax(1) == y).float().mean()
        return loss.item(), acc.item()

class SimpleModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim*2), nn.ReLU(),
            nn.Linear(dim*2, dim), nn.ReLU(),
            nn.Linear(dim, n_classes)
        )

    def forward(self, x):
        return self.net(x)

model = SimpleModel()
trainer = MixedPrecisionTrainer(model, lr=1e-3, weight_decay=0.01)

x_train = torch.randn(256, 64)
y_train = torch.randint(0, 10, (256,))
x_val = torch.randn(64, 64)
y_val = torch.randint(0, 10, (64,))
dataset = torch.utils.data.TensorDataset(x_train, y_train)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

train_losses = []
val_accs = []
for epoch in range(20):
    epoch_loss = 0
    for x, y in loader:
        loss = trainer.train_step(x, y)
        epoch_loss += loss
    train_losses.append(epoch_loss / len(loader))

    val_loss, val_acc = trainer.evaluate(x_val, y_val)
    val_accs.append(val_acc)

print(f"=== 混合精度训练 ===")
print(f"设备: {trainer.device_type}")
print(f"精度: {trainer.dtype}")
print(f"Loss Scaling: {'BF16不需要' if trainer.use_bf16 else 'FP16需要GradScaler'}")
print(f"训练Loss: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}")
print(f"验证准确率: {val_accs[0]:.4f} -> {val_accs[-1]:.4f}")

print(f"\n--- 显存估算 (7B模型) ---")
P = 7e9
configs = [
    ('FP32全量', 20*P),
    ('FP16混合精度', 16*P),
    ('FP16+8bit优化器', 6*P),
    ('QLoRA(NF4+LoRA)', 0.5*P + 0.01*P*16),
]
for name, mem in configs:
    print(f"  {name:<25} {mem/1e9:.1f} GB")

print(f"\n混合精度训练关键:")
print(f"1. GPU: FP16+GradScaler 或 BF16(无需Scaler)")
print(f"2. CPU: 仅支持BF16，不支持FP16")
print(f"3. Master Weights: 始终FP32，确保数值精度")
print(f"4. 梯度裁剪: unscale后裁剪，防止梯度爆炸")
print(f"5. 验证集监控: 防止混合精度训练中的数值不稳定")

### 梯度检查点（Gradient Checkpointing）

#### 什么是梯度检查点？

在前向传播时不保存所有中间激活值，只保存部分检查点层的激活。反向传播时从最近的检查点重新计算中间激活，用计算换显存。

#### 为什么用梯度检查点？

1. **显存大幅降低**：激活值显存降低约$k$倍（$k$为检查点间隔）
2. **训练大模型必备**：7B模型训练时激活值可能占数十GB显存
3. **PyTorch内置支持**：`torch.utils.checkpoint.checkpoint`

#### 梯度检查点的数学原理

标准训练：保存所有层激活，$A_{\text{total}} = \sum_{i=1}^{L} A_i$

梯度检查点：每$k$层保存一个检查点：
$$A_{\text{ckpt}} = \sum_{i \in \text{checkpoints}} A_i + A_{\text{current segment}}$$

显存节省：$\frac{A_{\text{ckpt}}}{A_{\text{total}}} \approx \frac{1}{k} + \frac{1}{L}$
计算开销：增加约1次前向传播（约33%额外计算）

#### 产业级实践

- PyTorch: `torch.utils.checkpoint.checkpoint`
- HuggingFace: `model.gradient_checkpointing_enable()`
- 典型配置：每层一个检查点（$k=1$），显存降低$\sqrt{L}$倍
- 注意：检查点内不要使用dropout随机性（`use_reentrant=False`推荐）

In [ ]:
from torch.utils.checkpoint import checkpoint

class CheckpointedTransformer(nn.Module):
    """带梯度检查点的Transformer - 产业级实现"""
    def __init__(self, dim=64, n_layers=8, n_heads=4, use_checkpoint=True):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(dim)

    def _layer_forward(self, layer, x):
        return layer(x)

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint and self.training:
                x = checkpoint(self._layer_forward, layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.norm(x)

dim = 64
model_no_ckpt = CheckpointedTransformer(dim, n_layers=8, use_checkpoint=False)
model_with_ckpt = CheckpointedTransformer(dim, n_layers=8, use_checkpoint=True)
model_with_ckpt.load_state_dict(model_no_ckpt.state_dict())

x = torch.randn(4, 32, dim, requires_grad=True)
x_ckpt = x.clone().detach().requires_grad_(True)

model_no_ckpt.train()
out_no_ckpt = model_no_ckpt(x)
loss_no_ckpt = out_no_ckpt.sum()
loss_no_ckpt.backward()

model_with_ckpt.train()
out_with_ckpt = model_with_ckpt(x_ckpt)
loss_with_ckpt = out_with_ckpt.sum()
loss_with_ckpt.backward()

output_diff = (out_no_ckpt - out_with_ckpt).abs().max().item()
grad_diff = (x.grad - x_ckpt.grad).abs().max().item()

print(f"=== 梯度检查点 ===")
print(f"层数: 8")
print(f"输出差异: {output_diff:.8f} (应接近0)")
print(f"梯度差异: {grad_diff:.8f} (应接近0)")

print(f"\n--- 激活值显存估算 (7B模型, 32层) ---")
activation_per_layer = 7e9 * 4 * 2048 / 1e9
total_activation = activation_per_layer * 32
ckpt_activation = activation_per_layer * (32 // 4 + 4)
print(f"  无检查点: ~{total_activation:.1f} GB (所有层激活)")
print(f"  每4层检查点: ~{ckpt_activation:.1f} GB")
print(f"  节省: {(1 - ckpt_activation/total_activation)*100:.0f}%")

print(f"\n--- HuggingFace集成 ---")
print(f"  model.gradient_checkpointing_enable()")
print(f"  自动在每层插入检查点，无需手动实现")
print(f"  配合LoRA使用: 检查点+LoRA = 端侧训练标配")

print(f"\n梯度检查点关键:")
print(f"1. 用计算换显存: 增加约33%计算时间")
print(f"2. PyTorch: torch.utils.checkpoint.checkpoint")
print(f"3. HuggingFace: model.gradient_checkpointing_enable()")
print(f"4. 推荐: use_reentrant=False (更安全)")
print(f"5. 注意: 检查点段内不要依赖随机性(如dropout)")

### 8bit优化器

#### 什么是8bit优化器？

将AdamW优化器的一阶矩$m$和二阶矩$v$从FP32量化为INT8/FP8，将优化器状态的显存占用从16字节/参数降至2字节/参数。

#### 为什么用8bit优化器？

1. **显存大幅降低**：7B模型优化器状态从112GB降至14GB
2. **训练精度几乎无损**：动态量化+分块归一化保持精度
3. **即插即用**：bitsandbytes的AdamW8bit与标准AdamW接口相同

#### 8bit优化器的数学原理

标准AdamW：$m_t, v_t \in \text{FP32}$，每参数16字节（$m$和$v$各4字节）

8bit AdamW：
1. 将$m_t$和$v_t$分块（block size=2048）
2. 每块计算绝对值最大值作为scale：$s_k = \max(|m_{t,k}|)$
3. 量化为INT8：$m_{t,k}^{\text{int8}} = \text{round}(m_{t,k} / s_k \cdot 127)$
4. 更新时反量化：$m_{t,k} = m_{t,k}^{\text{int8}} / 127 \cdot s_k$

每参数仅需2字节（INT8的$m$和$v$）+ 极少量scale元数据

#### 8bit优化器的局限性

- 需要CUDA GPU，CPU上不可用（bitsandbytes限制）
- 量化误差在小学习率场景下可能累积
- 推荐在显存紧张时使用，显存充足时标准AdamW更稳定

In [ ]:
class Adam8bit:
    """8bit AdamW优化器 - 教学实现
    使用分块INT8量化存储优化器状态，模拟bitsandbytes的AdamW8bit。
    产业级实现请使用bitsandbytes的AdamW8bit（需要CUDA GPU）。"""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, block_size=2048):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.block_size = block_size
        self.state = {}
        self.step_count = 0

    def _quantize_block(self, tensor):
        flat = tensor.flatten()
        n_blocks = math.ceil(flat.numel() / self.block_size)
        padded = F.pad(flat, (0, n_blocks * self.block_size - flat.numel()))
        blocks = padded.reshape(n_blocks, self.block_size)
        scales = blocks.abs().max(dim=1).values.clamp(min=1e-12)
        quantized = (blocks / scales.unsqueeze(1) * 127).round().clamp(-128, 127).to(torch.int8)
        return quantized, scales

    def _dequantize_block(self, quantized, scales, original_size):
        blocks = quantized.float() / 127.0 * scales.unsqueeze(1)
        return blocks.flatten()[:original_size]

    def step(self):
        self.step_count += 1
        for p in self.params:
            if p.grad is None:
                continue
            grad = p.grad.data
            if p not in self.state:
                self.state[p] = {
                    'step': 0,
                    'exp_avg_q': None, 'exp_avg_scale': None, 'exp_avg_size': None,
                    'exp_avg_sq_q': None, 'exp_avg_sq_scale': None,
                    'exp_avg_fp32': torch.zeros_like(p.data),
                    'exp_avg_sq_fp32': torch.zeros_like(p.data),
                }
            state = self.state[p]
            state['step'] += 1

            state['exp_avg_fp32'].mul_(self.beta1).add_(grad, alpha=1 - self.beta1)
            state['exp_avg_sq_fp32'].mul_(self.beta2).addcmul_(grad, grad, value=1 - self.beta2)

            exp_avg_q, exp_avg_scale = self._quantize_block(state['exp_avg_fp32'])
            exp_avg_sq_q, exp_avg_sq_scale = self._quantize_block(state['exp_avg_sq_fp32'])

            state['exp_avg_q'] = exp_avg_q
            state['exp_avg_scale'] = exp_avg_scale
            state['exp_avg_size'] = p.numel()
            state['exp_avg_sq_q'] = exp_avg_sq_q
            state['exp_avg_sq_scale'] = exp_avg_sq_scale

            exp_avg_deq = self._dequantize_block(exp_avg_q, exp_avg_scale, p.numel()).reshape(p.shape)
            exp_avg_sq_deq = self._dequantize_block(exp_avg_sq_q, exp_avg_sq_scale, p.numel()).reshape(p.shape)

            bias_correction1 = 1 - self.beta1 ** state['step']
            bias_correction2 = 1 - self.beta2 ** state['step']
            step_size = self.lr / bias_correction1
            denom = (exp_avg_sq_deq.sqrt() / math.sqrt(bias_correction2)).add_(self.eps)
            p.data.add_(exp_avg_deq / denom, alpha=-step_size)
            p.data.add_(p.data, alpha=-self.lr * self.weight_decay)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad = None

    def memory_footprint(self):
        total_bytes = 0
        for p, state in self.state.items():
            if state['exp_avg_q'] is not None:
                total_bytes += state['exp_avg_q'].numel() * 1
                total_bytes += state['exp_avg_scale'].numel() * 4
                total_bytes += state['exp_avg_sq_q'].numel() * 1
                total_bytes += state['exp_avg_sq_scale'].numel() * 4
        return total_bytes

model_8bit = SimpleModel()
optimizer_8bit = Adam8bit(model_8bit.parameters(), lr=1e-3)

model_fp32 = SimpleModel()
optimizer_fp32 = torch.optim.AdamW(model_fp32.parameters(), lr=1e-3)

x = torch.randn(64, 64)
y = torch.randint(0, 10, (64,))

losses_8bit, losses_fp32 = [], []
for _ in range(20):
    loss_8b = F.cross_entropy(model_8bit(x), y)
    optimizer_8bit.zero_grad()
    loss_8b.backward()
    optimizer_8bit.step()
    losses_8bit.append(loss_8b.item())

    loss_fp = F.cross_entropy(model_fp32(x), y)
    optimizer_fp32.zero_grad()
    loss_fp.backward()
    optimizer_fp32.step()
    losses_fp32.append(loss_fp.item())

fp32_opt_mem = sum(
    v.numel() * v.element_size()
    for s in optimizer_fp32.state.values()
    for v in s.values() if isinstance(v, torch.Tensor)
)

print(f"=== 8bit优化器 ===")
print(f"8bit AdamW Loss: {losses_8bit[0]:.4f} -> {losses_8bit[-1]:.4f}")
print(f"FP32 AdamW Loss: {losses_fp32[0]:.4f} -> {losses_fp32[-1]:.4f}")
print(f"\n优化器显存占用:")
print(f"  FP32 AdamW: {fp32_opt_mem/1024:.1f} KB")
print(f"  8bit AdamW(量化存储): {optimizer_8bit.memory_footprint()/1024:.1f} KB")
print(f"  节省: {(1 - optimizer_8bit.memory_footprint()/fp32_opt_mem)*100:.0f}%")

print(f"\n--- 7B模型优化器显存对比 ---")
P = 7e9
opt_configs = [
    ('AdamW (FP32)', 16*P/1e9),
    ('AdamW8bit (INT8)', 2*P/1e9),
    ('AdaFactor', 4*P/1e9),
    ('SGD (FP32)', 4*P/1e9),
]
for name, mem in opt_configs:
    print(f"  {name:<25} {mem:.1f} GB")

print(f"\n8bit优化器关键:")
print(f"1. bitsandbytes: AdamW8bit (需要CUDA GPU)")
print(f"2. 分块INT8量化: block_size=2048, 每块独立scale")
print(f"3. 显存: 优化器状态从16字节/参数降至2字节/参数")
print(f"4. 精度: 动态量化保持训练精度接近FP32")
print(f"5. CPU替代: AdaFactor (无需CUDA, 但优化器状态更大)")

### CPU Offloading与DeepSpeed ZeRO

#### 什么是CPU Offloading？

将优化器状态和参数卸载到CPU内存，仅在需要时传输到GPU。使大模型训练突破GPU显存限制。

#### 什么是DeepSpeed ZeRO？

ZeRO（Zero Redundancy Optimizer）通过划分优化器状态、梯度和参数来消除数据并行中的冗余：
- **ZeRO-1**：划分优化器状态（$m$和$v$），4倍显存节省
- **ZeRO-2**：ZeRO-1 + 划分梯度，8倍显存节省
- **ZeRO-3**：ZeRO-2 + 划分参数，$N$倍显存节省（$N$为GPU数）

#### CPU Offloading的数学原理

GPU显存需求：
$$M_{\text{GPU}} = M_{\text{active params}} + M_{\text{active grads}} + M_{\text{activations}}$$

CPU内存需求：
$$M_{\text{CPU}} = M_{\text{all params}} + M_{\text{optimizer states}}$$

数据传输：参数在需要时从CPU传输到GPU，梯度在反向传播后传回CPU
传输延迟：$T_{\text{transfer}} = \frac{M}{\text{PCIe BW}} \approx \frac{M}{12 \text{ GB/s}}$

#### 产业级配置

| 配置 | GPU显存(7B) | CPU内存(7B) | 训练速度 | 适用场景 |
|------|------------|------------|---------|----------|
| 全量GPU | ~140GB | ~0 | 1.0x | 8×A100 |
| ZeRO-2 | ~18GB | ~0 | 0.9x | 1×A100 |
| ZeRO-2+Offload | ~8GB | ~120GB | 0.5x | 1×RTX3090 |
| ZeRO-3+Offload | ~4GB | ~140GB | 0.3x | 1×RTX3060 |
| QLoRA | ~12GB | ~0 | 0.8x | 1×RTX3090 |

In [ ]:
class CPUOffloadOptimizer:
    """CPU Offload优化器 - 产业级实现
    优化器状态存储在CPU，参数更新时传输到GPU"""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.state = {}
        for p in self.params:
            self.state[p] = {
                'step': 0,
                'exp_avg': torch.zeros_like(p.data, device='cpu'),
                'exp_avg_sq': torch.zeros_like(p.data, device='cpu'),
            }

    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            state = self.state[p]
            state['step'] += 1
            grad_cpu = p.grad.data.cpu()
            state['exp_avg'].mul_(self.beta1).add_(grad_cpu, alpha=1 - self.beta1)
            state['exp_avg_sq'].mul_(self.beta2).addcmul_(grad_cpu, grad_cpu, value=1 - self.beta2)
            bias_correction1 = 1 - self.beta1 ** state['step']
            bias_correction2 = 1 - self.beta2 ** state['step']
            step_size = self.lr / bias_correction1
            denom = (state['exp_avg_sq'].sqrt() / math.sqrt(bias_correction2)).add_(self.eps)
            update = state['exp_avg'] / denom * step_size
            p.data.sub_(update.to(p.device))
            p.data.add_(p.data, alpha=-self.lr * self.weight_decay)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad = None

    def cpu_memory_mb(self):
        total = 0
        for state in self.state.values():
            total += state['exp_avg'].numel() * state['exp_avg'].element_size()
            total += state['exp_avg_sq'].numel() * state['exp_avg_sq'].element_size()
        return total / 1024 / 1024

model = SimpleModel()
offload_opt = CPUOffloadOptimizer(model.parameters(), lr=1e-3)

x = torch.randn(64, 64)
y = torch.randint(0, 10, (64,))

losses = []
for _ in range(10):
    loss = F.cross_entropy(model(x), y)
    offload_opt.zero_grad()
    loss.backward()
    offload_opt.step()
    losses.append(loss.item())

print(f"=== CPU Offload优化器 ===")
print(f"优化器状态存储: CPU")
print(f"CPU内存占用: {offload_opt.cpu_memory_mb():.2f} MB")
print(f"GPU仅存储: 参数+梯度+激活值")
print(f"训练Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

print(f"\n=== 端侧训练优化总结 ===")
print(f"\n{'优化技术':<20} {'显存节省':<15} {'计算开销':<15} {'适用场景':<20}")
print("-" * 70)
techniques = [
    ('混合精度(FP16/BF16)', '约50%', '几乎无', 'GPU/CPU训练'),
    ('梯度检查点', '约60-80%', '增加约33%', '大模型训练'),
    ('8bit优化器', '约75%优化器', '几乎无', 'GPU训练'),
    ('CPU Offload', '优化器→CPU', 'PCIe传输延迟', '单GPU大模型'),
    ('DeepSpeed ZeRO-2', '约8x', '通信开销', '多GPU训练'),
    ('DeepSpeed ZeRO-3', '约Nx', '通信+传输', '超大模型'),
    ('QLoRA', '约95%', '反量化开销', '端侧微调'),
]
for name, saving, overhead, scene in techniques:
    print(f"{name:<20} {saving:<15} {overhead:<15} {scene:<20}")

print(f"\n产业级端侧训练配置推荐:")
print(f"  RTX 4090 (24GB): QLoRA (NF4+LoRA) + 8bit优化器")
print(f"  RTX 3090 (24GB): QLoRA + 梯度检查点 + CPU Offload")
print(f"  RTX 3060 (12GB): QLoRA (小rank) + CPU Offload")
print(f"  Apple M2 (16GB): QLoRA + MPS加速 + 梯度检查点")
print(f"  CPU-only: QLoRA + BF16混合精度 + AdaFactor")

del model, offload_opt
gc.collect()

### 量化感知训练(QAT)

#### 什么是QAT？

量化感知训练（Quantization-Aware Training）在训练过程中插入伪量化节点，模拟量化带来的精度损失，让模型权重适应低精度表示。训练完成后，模型可以直接量化部署，精度损失远小于训练后量化(PTQ)。

#### 为什么用QAT？

1. **精度损失更小**：QAT的INT8精度损失通常<1%，而PTQ可能>3%
2. **适应量化噪声**：训练过程中模型学会了抵抗量化误差
3. **端侧部署必备**：端侧必须量化，QAT保证量化后精度

#### QAT的数学原理

伪量化（Fake Quantization）：前向传播时模拟量化，反向传播时使用STE（Straight-Through Estimator）传递梯度

$$\bar{W} = \text{dequant}(\text{quant}(W, b), b)$$

其中$\text{quant}$将FP32权重映射到$2^b$个量化级别，$\text{dequant}$映射回FP32。

STE梯度估计：
$$\frac{\partial L}{\partial W} \approx \frac{\partial L}{\partial \bar{W}}$$

即梯度直接"穿透"量化操作，忽略量化函数的不可导性。

#### QAT vs PTQ

| 方法 | 训练精度 | 部署精度 | 精度损失 | 训练开销 |
|------|---------|---------|---------|----------|
| **PTQ** (训练后量化) | FP32/FP16 | INT8/INT4 | 较大 | 无额外开销 |
| **QAT** (量化感知训练) | FP32+伪量化 | INT8/INT4 | 小 | +20-30%训练时间 |
| **QLoRA** | NF4基座+FP16 LoRA | 重新量化 | 中等 | 基座无开销 |

In [ ]:
class FakeQuantize(nn.Module):
    """伪量化模块 - QAT核心组件
    前向: 量化+反量化(模拟量化误差)
    反向: STE (Straight-Through Estimator) 梯度直通"""
    def __init__(self, num_bits=8, symmetric=True, per_channel=False):
        super().__init__()
        self.num_bits = num_bits
        self.symmetric = symmetric
        self.per_channel = per_channel
        self.scale = None
        self.zero_point = None
        self.qmin = -(2 ** (num_bits - 1)) if symmetric else 0
        self.qmax = 2 ** (num_bits - 1) - 1 if symmetric else 2 ** num_bits - 1
        self.register_buffer('observer_min', torch.tensor(float('inf')))
        self.register_buffer('observer_max', torch.tensor(float('-inf')))

    def forward(self, x):
        if self.training:
            with torch.no_grad():
                if self.per_channel:
                    self.observer_min.data = torch.min(self.observer_min, x.min(dim=1, keepdim=True).values)
                    self.observer_max.data = torch.max(self.observer_max, x.max(dim=1, keepdim=True).values)
                else:
                    self.observer_min.data = torch.min(self.observer_min, x.min())
                    self.observer_max.data = torch.max(self.observer_max, x.max())

        if self.symmetric:
            scale = torch.max(self.observer_min.abs(), self.observer_max.abs()) / self.qmax
            scale = torch.clamp(scale, min=1e-8)
            zero_point = torch.zeros_like(scale)
        else:
            scale = (self.observer_max - self.observer_min) / (self.qmax - self.qmin)
            scale = torch.clamp(scale, min=1e-8)
            zero_point = self.qmin - torch.round(self.observer_min / scale)

        self.scale = scale
        self.zero_point = zero_point
        x_q = torch.clamp(torch.round(x / scale + zero_point), self.qmin, self.qmax)
        x_dq = (x_q - zero_point) * scale
        return x + (x_dq - x).detach()

class QATLinear(nn.Module):
    """QAT线性层: 权重伪量化 + FP32计算"""
    def __init__(self, in_features, out_features, num_bits=8):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.weight_quantizer = FakeQuantize(num_bits, symmetric=True, per_channel=True)

    def forward(self, x):
        w_q = self.weight_quantizer(self.weight)
        return F.linear(x, w_q, self.bias)

class QATModel(nn.Module):
    """QAT微调模型"""
    def __init__(self, dim=64, n_classes=10, num_bits=8):
        super().__init__()
        self.fc1 = QATLinear(dim, dim*2, num_bits)
        self.fc2 = QATLinear(dim*2, dim, num_bits)
        self.fc3 = QATLinear(dim, n_classes, num_bits)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

dim, n_classes = 64, 10
qat_model = QATModel(dim, n_classes, num_bits=8)
fp32_model = SimpleModel(dim, n_classes)
fp32_model.load_state_dict({
    k.replace('net.', '').replace('0.', 'fc1.').replace('2.', 'fc2.').replace('4.', 'fc3.'): v
    for k, v in SimpleModel(dim, n_classes).state_dict().items()
}, strict=False)

x_data = torch.randn(256, dim)
y_data = torch.randint(0, n_classes, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

qat_opt = torch.optim.AdamW(qat_model.parameters(), lr=1e-3)
fp32_opt = torch.optim.AdamW(fp32_model.parameters(), lr=1e-3)

qat_losses, fp32_losses = [], []
for epoch in range(20):
    for x, y in loader:
        qat_loss = F.cross_entropy(qat_model(x), y)
        qat_opt.zero_grad()
        qat_loss.backward()
        qat_opt.step()
        qat_losses.append(qat_loss.item())

        fp32_loss = F.cross_entropy(fp32_model(x), y)
        fp32_opt.zero_grad()
        fp32_loss.backward()
        fp32_opt.step()
        fp32_losses.append(fp32_loss.item())

with torch.no_grad():
    qat_acc = (qat_model(x_data).argmax(1) == y_data).float().mean()
    fp32_acc = (fp32_model(x_data).argmax(1) == y_data).float().mean()

print(f"=== 量化感知训练(QAT) ===")
print(f"\n训练Loss对比:")
print(f"  QAT (INT8伪量化): {qat_losses[0]:.4f} -> {qat_losses[-1]:.4f}")
print(f"  FP32 (无量化):     {fp32_losses[0]:.4f} -> {fp32_losses[-1]:.4f}")
print(f"\n训练集准确率:")
print(f"  QAT: {qat_acc:.4f}")
print(f"  FP32: {fp32_acc:.4f}")

print(f"\n--- QAT量化误差分析 ---")
qat_model.eval()
with torch.no_grad():
    for name, module in qat_model.named_modules():
        if isinstance(module, QATLinear):
            w_orig = module.weight.data
            w_q = module.weight_quantizer(w_orig)
            q_error = (w_orig - w_q).abs().mean().item()
            q_range = (w_orig - w_q).abs().max().item()
            print(f"  {name}: 平均量化误差={q_error:.6f}, 最大误差={q_range:.6f}")

print(f"\nQAT关键:")
print(f"1. 伪量化: 前向模拟量化误差，反向STE梯度直通")
print(f"2. 对称量化: 权重量化通常使用对称量化(per_channel)")
print(f"3. 观察器: 训练中收集权重范围，用于确定量化参数")
print(f"4. PyTorch: torch.quantization.prepare/convert")
print(f"5. 产业级: PyTorch 2.x + torch.ao.quantization")
print(f"6. QAT训练后直接转INT8部署，精度损失<1%")

### 知识蒸馏

#### 什么是知识蒸馏？

知识蒸馏（Knowledge Distillation）使用大模型（教师）的输出指导小模型（学生）训练，让学生模型学习教师模型的"暗知识"（soft labels）。

#### 为什么用知识蒸馏？

1. **小模型精度提升**：蒸馏后的小模型精度接近大模型
2. **端侧部署友好**：小模型适合端侧推理
3. **保留暗知识**：教师模型的输出分布比hard label信息更丰富

#### 知识蒸馏的数学原理

蒸馏损失 = 硬标签损失 + 软标签损失：
$$L = (1-\alpha) \cdot L_{\text{CE}}(y, \hat{y}) + \alpha \cdot T^2 \cdot L_{\text{KL}}(\sigma(z_t/T), \sigma(z_s/T))$$

其中：
- $z_t, z_s$：教师和学生的logits
- $T$：温度参数（通常3-5），$T$越大软标签越平滑
- $\alpha$：蒸馏损失权重（通常0.5-0.9）
- $T^2$：补偿因子，因为KL散度在高温下缩小约$T^2$倍

#### 蒸馏方式

| 蒸馏方式 | 教师 | 学生 | 适用场景 |
|---------|------|------|----------|
| **离线蒸馏** | 大模型(云端) | 小模型(端侧) | 预训练阶段 |
| **在线蒸馏** | 大模型+小模型同步训练 | 小模型 | 微调阶段 |
| **自蒸馏** | 模型自身EMA | 模型自身 | 端侧持续学习 |
| **特征蒸馏** | 教师中间层特征 | 学生中间层 | 结构化知识迁移 |

In [ ]:
class DistillationTrainer:
    """知识蒸馏训练器 - 产业级实现
    支持离线蒸馏(固定教师)和自蒸馏(EMA教师)"""
    def __init__(self, teacher, student, lr=1e-3, temperature=4.0, alpha=0.7,
                 weight_decay=0.01):
        self.teacher = teacher
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        self.optimizer = torch.optim.AdamW(
            student.parameters(), lr=lr, weight_decay=weight_decay
        )

        for p in self.teacher.parameters():
            p.requires_grad = False
        self.teacher.eval()

    def distillation_loss(self, student_logits, teacher_logits, labels):
        hard_loss = F.cross_entropy(student_logits, labels)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=1)
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=1)
        soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (self.temperature ** 2)
        total_loss = (1 - self.alpha) * hard_loss + self.alpha * soft_loss
        return total_loss, hard_loss, soft_loss

    def train_step(self, x, y):
        self.student.train()
        self.optimizer.zero_grad()

        student_logits = self.student(x)
        with torch.no_grad():
            teacher_logits = self.teacher(x)

        total_loss, hard_loss, soft_loss = self.distillation_loss(
            student_logits, teacher_logits, y
        )
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.student.parameters(), max_norm=1.0)
        self.optimizer.step()

        return total_loss.item(), hard_loss.item(), soft_loss.item()

class TeacherModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim*4), nn.ReLU(),
            nn.Linear(dim*4, dim*2), nn.ReLU(),
            nn.Linear(dim*2, dim), nn.ReLU(),
            nn.Linear(dim, n_classes)
        )

    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(),
            nn.Linear(dim, n_classes)
        )

    def forward(self, x):
        return self.net(x)

dim, n_classes = 64, 10
teacher = TeacherModel(dim, n_classes)
student = StudentModel(dim, n_classes)
student_no_distill = StudentModel(dim, n_classes)

x_data = torch.randn(512, dim)
y_data = torch.randint(0, n_classes, (512,))
x_val = torch.randn(128, dim)
y_val = torch.randint(0, n_classes, (128,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

teacher_opt = torch.optim.AdamW(teacher.parameters(), lr=1e-3)
for _ in range(50):
    for x, y in loader:
        loss = F.cross_entropy(teacher(x), y)
        teacher_opt.zero_grad()
        loss.backward()
        teacher_opt.step()

distill_trainer = DistillationTrainer(teacher, student, lr=1e-3, temperature=4.0, alpha=0.7)
no_distill_opt = torch.optim.AdamW(student_no_distill.parameters(), lr=1e-3)

distill_losses, no_distill_losses = [], []
distill_accs, no_distill_accs = [], []
for epoch in range(20):
    for x, y in loader:
        total, hard, soft = distill_trainer.train_step(x, y)
        distill_losses.append(total)

        loss_nd = F.cross_entropy(student_no_distill(x), y)
        no_distill_opt.zero_grad()
        loss_nd.backward()
        no_distill_opt.step()
        no_distill_losses.append(loss_nd.item())

    with torch.no_grad():
        d_acc = (student(x_val).argmax(1) == y_val).float().mean().item()
        nd_acc = (student_no_distill(x_val).argmax(1) == y_val).float().mean().item()
        t_acc = (teacher(x_val).argmax(1) == y_val).float().mean().item()
        distill_accs.append(d_acc)
        no_distill_accs.append(nd_acc)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print(f"=== 知识蒸馏 ===")
print(f"\n模型参数量:")
print(f"  教师模型: {teacher_params:,}")
print(f"  学生模型: {student_params:,} ({student_params/teacher_params:.1%} of teacher)")

print(f"\n验证集准确率:")
print(f"  教师模型: {t_acc:.4f}")
print(f"  蒸馏学生: {distill_accs[-1]:.4f}")
print(f"  无蒸馏学生: {no_distill_accs[-1]:.4f}")
print(f"  蒸馏提升: +{(distill_accs[-1] - no_distill_accs[-1])*100:.1f}%")

print(f"\n蒸馏配置:")
print(f"  温度T=4.0: 软标签更平滑，暗知识更丰富")
print(f"  alpha=0.7: 70%权重给软标签损失")
print(f"  T^2补偿: KL散度乘以T^2=16补偿高温缩放")

print(f"\n知识蒸馏产业实践:")
print(f"1. 离线蒸馏: 云端大模型→端侧小模型，预训练阶段")
print(f"2. 自蒸馏: 端侧模型EMA作为教师，持续学习")
print(f"3. 特征蒸馏: 中间层特征对齐，更细粒度的知识迁移")
print(f"4. LoRA+蒸馏: 用大模型指导LoRA微调，提升微调效果")

### 产业级实战：端侧QLoRA微调

结合前面所有优化技术，使用bitsandbytes + PEFT在GPT-2上演示端侧QLoRA微调的完整流程。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=nf4_config,
    device_map='auto' if torch.cuda.is_available() else None,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=['c_attn'],
    bias='none',
)

peft_model = get_peft_model(model_4bit, lora_config)
peft_model.print_trainable_parameters()

train_texts = [
    'On-device AI enables privacy-preserving inference on edge devices.',
    'Quantization reduces model size with minimal accuracy loss for deployment.',
    'LoRA enables efficient fine-tuning by training only low-rank adapters.',
    'Mixed precision training uses FP16 for forward and FP32 for optimizer.',
    'Gradient checkpointing trades computation for memory by recomputing activations.',
]

encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=48, return_tensors='pt')
dataset = torch.utils.data.TensorDataset(encodings['input_ids'], encodings['attention_mask'])
loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

peft_model.train()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, peft_model.parameters()), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

print(f"\n=== 端侧QLoRA微调 ===")
for epoch in range(3):
    total_loss = 0
    for batch_ids, batch_mask in loader:
        with torch.autocast(device_type='cpu', dtype=torch.bfloat16):
            outputs = peft_model(input_ids=batch_ids, attention_mask=batch_mask, labels=batch_ids)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(peft_model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    scheduler.step()
    print(f"  Epoch {epoch+1}/3, Loss: {total_loss/len(loader):.4f}")

peft_model.eval()
test_input = tokenizer('On-device AI enables', return_tensors='pt')
with torch.no_grad():
    output = peft_model.generate(**test_input, max_new_tokens=20, do_sample=False)
print(f"\n生成结果: {tokenizer.decode(output[0], skip_special_tokens=True)}")

print(f"\n端侧QLoRA微调完整流程:")
print(f"1. BitsAndBytesConfig: NF4量化+双重量化+BF16计算")
print(f"2. LoraConfig: rank=8, alpha=16, target=c_attn")
print(f"3. 混合精度: BF16前向+FP32优化器")
print(f"4. 优化器: AdamW (仅训练LoRA参数)")
print(f"5. 合并: merge_and_unload() → 部署")

del peft_model, model_4bit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
### 端侧训练的特有挑战

端侧训练与云端训练有本质区别，面临独特的硬件和环境约束。

#### 端侧训练 vs 云端训练

| 维度 | 云端训练 | 端侧训练 |
|------|---------|----------|
| **算力** | A100/H100, TFLOPS级 | 手机NPU, TOPS级 |
| **内存** | 80GB HBM | 4-16GB DRAM |
| **功耗** | 300W+ | 5-10W |
| **散热** | 数据中心空调 | 被动散热，易过热 |
| **数据** | 集中式，大规模 | 分布式，小批量 |
| **训练时间** | 小时-天 | 秒-分钟 |
| **目标** | 从零训练 | 微调/适配 |

#### 热节流（Thermal Throttling）

端侧设备被动散热，持续高负载会导致芯片降频：

$$T_{\text{chip}}(t) = T_{\text{ambient}} + P \cdot R_{\theta} \cdot (1 - e^{-t/\tau})$$

其中$R_{\theta}$为热阻，$\tau$为热时间常数。当$T_{\text{chip}} > T_{\text{threshold}}$时，芯片降频保护。

- 骁龙8 Gen3: 持续推理5分钟后NPU可能降频30-50%
- Apple A17 Pro: 持续GPU负载3分钟后降频
- **对策**：间歇训练（训练-冷却交替）、降低batch size、使用低功耗计算单元

#### 内存碎片化

端侧训练时，KV Cache、梯度、优化器状态、激活值共存，容易导致内存碎片：

| 内存消费者 | 推理时 | 训练时 |
|-----------|--------|--------|
| **模型权重** | 3.5GB(INT4) | 3.5GB(INT4, 冻结) |
| **KV Cache** | 200MB | 200MB |
| **梯度** | 0 | 20MB(仅LoRA) |
| **优化器状态** | 0 | 40MB(Adam, 仅LoRA) |
| **激活值** | 0 | 500MB(需反向传播) |
| **碎片开销** | 10% | 20-30% |

**对策**：
1. 梯度检查点（Gradient Checkpointing）：用计算换内存，仅保存部分激活值
2. 预分配内存池：避免动态分配导致的碎片
3. 选择性反向传播：仅对LoRA层计算梯度，冻结层不保存激活值

In [ ]:
class OnDeviceTrainingAnalyzer:
    """端侧训练分析器 - 产业级实现"""
    def __init__(self, model_params_b=7, lora_ratio=0.001, dtype='int4_base_fp16_lora'):
        self.model_params = model_params_b * 1e9
        self.lora_params = self.model_params * lora_ratio
        self.dtype = dtype

    def estimate_memory(self, seq_len=512, n_layers=32, hidden_dim=4096,
                        use_gradient_checkpoint=False, use_8bit_optimizer=False):
        if self.dtype == 'int4_base_fp16_lora':
            base_bytes = 0.5
            lora_bytes = 2
        else:
            base_bytes = 2
            lora_bytes = 2

        base_weight_mb = self.model_params * base_bytes / 1e6
        lora_weight_mb = self.lora_params * lora_bytes / 1e6
        lora_grad_mb = lora_weight_mb
        optimizer_bytes_per_param = 2 if use_8bit_optimizer else 8
        optimizer_mb = lora_weight_mb * (optimizer_bytes_per_param / 2)
        kv_cache_mb = seq_len * n_layers * 2 * 32 * 128 * 2 / 1e6
        activation_mb = seq_len * hidden_dim * n_layers * 2 / 1e6
        if use_gradient_checkpoint:
            activation_mb = activation_mb * 0.25
        fragment_mb = (base_weight_mb + lora_weight_mb + lora_grad_mb + optimizer_mb + kv_cache_mb + activation_mb) * 0.2
        total = base_weight_mb + lora_weight_mb + lora_grad_mb + optimizer_mb + kv_cache_mb + activation_mb + fragment_mb

        return {
            'base_weight_mb': base_weight_mb, 'lora_weight_mb': lora_weight_mb,
            'lora_grad_mb': lora_grad_mb, 'optimizer_mb': optimizer_mb,
            'kv_cache_mb': kv_cache_mb, 'activation_mb': activation_mb,
            'fragment_mb': fragment_mb, 'total_mb': total,
        }

    def simulate_thermal(self, power_w=8, duration_s=300, threshold_c=85, ambient_c=35,
                         r_theta=15, tau=120, throttle_pct=0.5):
        temps = []
        current_power = power_w
        for t in range(0, duration_s, 10):
            temp = ambient_c + current_power * r_theta * (1 - np.exp(-t / tau))
            throttled = temp > threshold_c
            if throttled:
                current_power = power_w * throttle_pct
                temp = threshold_c + (temp - threshold_c) * 0.3
            temps.append({'time_s': t, 'temp_c': temp, 'throttled': throttled,
                          'effective_power_w': current_power})
        return temps

analyzer = OnDeviceTrainingAnalyzer()

configs = [
    ('基线(无优化)', False, False),
    ('+梯度检查点', True, False),
    ('+8bit优化器', False, True),
    ('+检查点+8bit', True, True),
]

print("=== 端侧训练内存分析 (7B INT4基座 + LoRA微调) ===")
print(f"\n{'配置':<20} {'总内存(MB)':<12} {'激活值(MB)':<12} {'优化器(MB)':<12} {'适合设备'}")
print("-" * 70)
for name, use_ckpt, use_8bit in configs:
    mem = analyzer.estimate_memory(use_gradient_checkpoint=use_ckpt, use_8bit_optimizer=use_8bit)
    device = '✓ 8GB' if mem['total_mb'] < 6144 else '✓ 12GB' if mem['total_mb'] < 10240 else '✗ 需16GB+'
    print(f"{name:<20} {mem['total_mb']:<12.0f} {mem['activation_mb']:<12.0f} {mem['optimizer_mb']:<12.0f} {device}")

temps = analyzer.simulate_thermal()
print(f"\n=== 热节流模拟 (8W功耗, 被动散热) ===")
for t in temps[::3]:
    marker = ' ⚠ 降频!' if t['throttled'] else ''
    print(f"  {t['time_s']:>4d}s: {t['temp_c']:.1f}°C (功耗: {t['effective_power_w']:.1f}W){marker}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 端侧训练仅适合LoRA微调: 全量微调内存不够")
print(f"2. 梯度检查点: 激活值减少75%，端侧训练标配")
print(f"3. 8bit优化器: 进一步减少优化器内存")
print(f"4. 热节流: 间歇训练是标准方案(30秒训练+30秒冷却)")
print(f"5. 内存碎片: 预分配内存池比动态分配更高效")

---
### 端侧持续学习与灾难性遗忘防护

端侧设备需要持续从用户数据中学习，但新数据训练会导致模型遗忘旧知识——这就是灾难性遗忘（Catastrophic Forgetting）。

#### 灾难性遗忘的数学描述

设旧任务数据分布为$D_{\text{old}}$，新任务数据分布为$D_{\text{new}}$，在$D_{\text{new}}$上微调后：

$$\mathcal{L}_{\text{old}}(\theta_{\text{new}}) \gg \mathcal{L}_{\text{old}}(\theta_{\text{old}})$$

即模型在新数据上训练后，在旧数据上的损失显著增大。

#### 防遗忘策略

| 策略 | 原理 | 内存开销 | 效果 | 适用场景 |
|------|------|---------|------|----------|
| **EWC** | Fisher信息矩阵正则化 | 中(存Fisher) | 中 | 少任务 |
| **LoRA回滚** | 仅更新LoRA，旧LoRA可回滚 | 低(存旧LoRA) | 高 | LoRA微调 |
| **经验回放** | 保留少量旧数据混合训练 | 中(存旧样本) | 高 | 通用 |
| **知识蒸馏** | 旧模型作为教师，约束输出 | 低 | 中高 | 模型更新 |
| **Progressive Networks** | 新任务添加新列 | 高 | 极高 | 少任务 |

#### EWC（Elastic Weight Consolidation）

EWC通过Fisher信息矩阵识别对旧任务重要的参数，限制这些参数的更新：

$$\mathcal{L}_{\text{EWC}} = \mathcal{L}_{\text{new}}(\theta) + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta_{\text{old},i})^2$$

其中$F_i$为Fisher信息矩阵的对角元素，衡量参数$\theta_i$对旧任务的重要性。

#### LoRA回滚：端侧持续学习的最佳实践

利用LoRA的可插拔特性，实现零遗忘的持续学习：

```
基座模型(W₀) ← 冻结，永不更新
  ├── LoRA_v1 (通用能力) ← 冻结
  ├── LoRA_v2 (用户偏好) ← 可更新
  └── LoRA_v3 (最新适配) ← 活跃训练

回滚: 删除LoRA_v3 → 恢复到v2状态
```

优点：
1. **零遗忘**：旧LoRA冻结，不受新训练影响
2. **快速回滚**：删除最新LoRA即可回滚
3. **内存高效**：每个LoRA仅20-50MB
4. **组合灵活**：不同LoRA可自由组合

In [ ]:
class ContinualLearningSimulator:
    """端侧持续学习模拟器 - 产业级实现"""
    def __init__(self, n_tasks=5, n_params=1000):
        self.n_tasks = n_tasks
        self.n_params = n_params
        np.random.seed(42)
        self.task_params = [np.random.randn(n_params) * 0.1 for _ in range(n_tasks)]
        self.fisher_info = [np.abs(np.random.randn(n_params)) for _ in range(n_tasks)]

    def simulate_naive_finetuning(self, lr=0.01, n_steps=100):
        theta = np.zeros(self.n_params)
        forgetting = []
        for task_id in range(self.n_tasks):
            for _ in range(n_steps):
                grad = -2 * (theta - self.task_params[task_id])
                theta += lr * grad
            old_losses = [np.mean((theta - self.task_params[i]) ** 2) for i in range(task_id + 1)]
            forgetting.append({
                'task': task_id, 'current_loss': old_losses[-1],
                'avg_old_loss': np.mean(old_losses[:-1]) if len(old_losses) > 1 else 0,
            })
        return forgetting

    def simulate_ewc(self, lr=0.01, n_steps=100, ewc_lambda=10.0):
        theta = np.zeros(self.n_params)
        old_thetas, fishers = [], []
        forgetting = []
        for task_id in range(self.n_tasks):
            for _ in range(n_steps):
                grad = -2 * (theta - self.task_params[task_id])
                ewc_grad = sum(f * (theta - ot) for ot, f in zip(old_thetas, fishers))
                grad += ewc_lambda * ewc_grad
                theta += lr * grad
            old_thetas.append(theta.copy())
            fishers.append(self.fisher_info[task_id].copy())
            old_losses = [np.mean((theta - self.task_params[i]) ** 2) for i in range(task_id + 1)]
            forgetting.append({
                'task': task_id, 'current_loss': old_losses[-1],
                'avg_old_loss': np.mean(old_losses[:-1]) if len(old_losses) > 1 else 0,
            })
        return forgetting

    def simulate_lora_rollback(self, lr=0.01, n_steps=100):
        base = np.zeros(self.n_params)
        loras = []
        forgetting = []
        for task_id in range(self.n_tasks):
            lora = np.zeros(self.n_params)
            target = self.task_params[task_id] - base - sum(loras)
            for _ in range(n_steps):
                grad = -2 * (lora - target)
                lora += lr * grad
            loras.append(lora.copy())
            theta = base + sum(loras)
            old_losses = [np.mean((theta - self.task_params[i]) ** 2) for i in range(task_id + 1)]
            forgetting.append({
                'task': task_id, 'current_loss': old_losses[-1],
                'avg_old_loss': np.mean(old_losses[:-1]) if len(old_losses) > 1 else 0,
            })
        return forgetting

cl_sim = ContinualLearningSimulator(n_tasks=5)
naive = cl_sim.simulate_naive_finetuning()
ewc = cl_sim.simulate_ewc()
lora_rb = cl_sim.simulate_lora_rollback()

print("=== 持续学习策略对比 (5个顺序任务) ===")
print(f"\n{'任务':<6} {'朴素-当前':<12} {'朴素-旧任务':<14} {'EWC-当前':<12} {'EWC-旧任务':<14} {'LoRA回滚-当前':<14} {'LoRA回滚-旧任务'}")
print("-" * 90)
for i in range(5):
    print(f"T{i:<4} {naive[i]['current_loss']:<12.4f} {naive[i]['avg_old_loss']:<14.4f} "
          f"{ewc[i]['current_loss']:<12.4f} {ewc[i]['avg_old_loss']:<14.4f} "
          f"{lora_rb[i]['current_loss']:<14.4f} {lora_rb[i]['avg_old_loss']:.4f}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 朴素微调: 严重遗忘，旧任务损失随任务数增加而飙升")
print(f"2. EWC: 减轻遗忘，但Fisher矩阵需要额外内存和计算")
print(f"3. LoRA回滚: 最佳方案，旧任务零遗忘(旧LoRA冻结)")
print(f"4. 端侧持续学习流程: 收集用户数据→LoRA微调→A/B测试→灰度发布")
print(f"5. 数据管理: 旧数据摘要(核心集)保留用于回放")
print(f"6. 评估: 每次更新后在保留的验证集上测试旧任务性能")

---
## 总结与最佳实践

### 端侧训练优化的核心原则

1. **混合精度是标配**：BF16训练+FP32优化器状态，显存减少50%+
2. **梯度检查点是内存救星**：用计算换内存，适合4-8GB设备
3. **8bit优化器是锦上添花**：进一步减少优化器状态内存
4. **CPU Offloading是最后手段**：延迟增加3-5x，仅极端内存不足时使用
5. **QAT保证量化精度**：训练后必须量化部署时，QAT是保证精度的关键
6. **知识蒸馏提升小模型**：用大模型指导端侧模型训练
7. **热管理是长期运行的保障**：端侧推理+训练必须在TDP内完成

### 端侧训练优化策略选择

| 场景 | 推荐策略 | 预期显存节省 |
|------|---------|------------|
| **8GB设备, 1.5B模型** | BF16 + QLoRA(r=8) | 基线可训练 |
| **4GB设备, 1.5B模型** | BF16 + QLoRA + 梯度检查点 | ~60% |
| **4GB设备, 3B模型** | BF16 + QLoRA + 检查点 + 8bit优化器 | ~75% |
| **2GB设备, 1.5B模型** | NF4 + QLoRA + 检查点 + CPU Offload | ~90% |

### 端侧训练后重新量化的流程

```
LoRA微调完成 → 合并LoRA权重 → 校准数据集 → PTQ量化 → 精度验证
                                    ↓ 失败
                              QAT微调1-2 epoch → 重新量化 → 精度验证
```

### 端侧训练Checklist

- [ ] 确定设备内存预算和模型大小
- [ ] 选择混合精度策略（BF16/FP16）
- [ ] 启用梯度检查点（内存紧张时）
- [ ] 选择优化器（AdamW/8bit Adam/Adafactor）
- [ ] 实现CPU Offloading（极端内存不足时）
- [ ] 热管理测试：长时间训练的温度和性能稳定性
- [ ] 训练后量化：校准数据集+PTQ，必要时QAT
- [ ] 知识蒸馏：用大模型指导端侧模型训练
- [ ] 持续学习策略：LoRA回滚防止灾难性遗忘
- [ ] 训练监控：loss曲线、梯度范数、内存使用
- [ ] 精度验证：微调后模型在目标任务和通用任务上的表现